与前后端/跨语言接口最通用 返回的是字典，类型不匹配的时候不抛出异常，并且比较繁琐，缺号校验机制，只做了解,也可以进行嵌套

举例1

In [1]:
from pydantic_core.core_schema import json_schema
from typing_extensions import TypedDict,Annotated
from typing import Literal, Optional
from langchain.chat_models import init_chat_model
from rich import print as rprint
##统一使用init_chat_model（）完成模型的初始化
import os
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
json_schema={
    "title":"Movie",
    "description":"电影的信息",
    "type": "object",
    "properties":{
        "title":{
            "type":"string",
            "description":"电影名称"
        },
        "director":{
            "type":"string",
            "description":"电影导演名称"
        },
        "year":{
            "type":"string",
            "description":"电影上映时间"
        },
        "rating":{
            "type":"integer",
            "description":"评分"
        }
    },
    "required":["title","director","year","rating"]

}



#1 读取，env文件中的配置信息，相关的环境变量以.env文件中的优先
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#2 初始化大模型
model= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)
structed_model=model.with_structured_output(json_schema,method="json_schema")
message=[HumanMessage("给我介绍一下《唐伯虎点秋香》的信息")]
response=structed_model.invoke(message)
print(response)

{'title': '《唐伯虎点秋香》', 'director': '李力持', 'year': '1993年', 'rating': 9}
